# Webserv - Guide de Correction

Analyse point par point du sujet de correction vs le code actuel.

Légende : <span style="color: green; font-weight: bold;">FAIT</span> | <span style="color: orange; font-weight: bold;">PARTIEL</span> | <span style="color: red; font-weight: bold;">MANQUANT</span> | <span style="color: dodgerblue; font-weight: bold;">A TESTER</span>

---
## 1. Check the code and ask questions

### Explain the basics of an HTTP server
<span style="color: green; font-weight: bold;">FAIT</span> — Le serveur gère le cycle complet : socket → bind → listen → accept → recv → parse → route → send → close.

### Ask which function they used for I/O Multiplexing
<span style="color: green; font-weight: bold;">FAIT</span> — `poll()` est utilisé (`Server.cpp:579`). Défini dans le Makefile comme `POLL_METHOD = poll` sur Linux.

### Ask to get an explanation of how select (or equivalent) is working
<span style="color: green; font-weight: bold;">FAIT</span> — `poll()` surveille un tableau de `struct pollfd` avec les events POLLIN/POLLOUT. Timeout de 1000ms.

### Ask if they use only one select (or equivalent)
<span style="color: green; font-weight: bold;">FAIT</span> — Un seul appel `poll()` dans la boucle principale `Server::run()` (ligne 579). Il gère à la fois :
- Les sockets serveur (accept)
- Les sockets clients (read/write)
- Les pipes CGI (read)

### The select should check fd for read and write AT THE SAME TIME
<span style="color: green; font-weight: bold;">FAIT</span> — Les sockets clients sont enregistrés avec `pfd.events = POLLIN | POLLOUT` (ligne 147). poll() vérifie les deux en même temps.

### There should be only one read or one write per client per select
<span style="color: green; font-weight: bold;">FAIT</span> — Dans `handleClientEvents()` : un seul `client->readData()` (un seul `recv`) et un seul `client->writeData()` (un seul `send`) par tour de boucle.

### Search for all read/recv/write/send and check error handling
<span style="color: green; font-weight: bold;">FAIT</span> — 
- `recv()` dans `Client::readData()` (Client.cpp:106) : vérifie `< 0` (erreur directe) et `== 0` (déconnexion)
- `send()` dans `Client::writeData()` (Client.cpp:160) : vérifie `n > 0`, `n < 0` (erreur) et `n == 0` (déconnexion)
- `read()` dans `Server::handleCGIEvents()` (Server.cpp:463) : boucle jusqu'à `n == 0`
- Client est supprimé dans `toRemove` quand erreur détectée

### Check if returned value is well checked (checking only -1 or 0 is not good)
<span style="color: green; font-weight: bold;">FAIT</span> —
- `recv()` : vérifie `< 0` ET `== 0` séparément → OK
- `send()` : vérifie `n > 0`, `n < 0` ET `n == 0` séparément → OK

### If a check of errno is done after read/recv/write/send → mark to 0
<span style="color: green; font-weight: bold;">FAIT</span> — Aucun check `errno` après `recv()`/`send()`. Le `readData()` traite `bytesRead < 0` comme erreur directe sans vérifier errno. poll() gère les erreurs via `POLLERR/POLLHUP`.

### Writing or reading ANY file descriptor without going through the select is FORBIDDEN
<span style="color: green; font-weight: bold;">FAIT</span> —
- Toutes les lectures/écritures clients passent par `poll()` → **OK**
- Les pipes CGI (pipe_out) passent par `poll()` (ajoutés dans `buildPollFds()`) → **OK**
- `startCGI()` écrit le body POST dans `pipe_in` puis le ferme immédiatement → c'est le setup du CGI (pipe d'entrée créé et fermé dans la même fonction), pas un fd surveillé par poll. Le `pipe_out` (lecture) passe bien par poll → **OK**

### The project must compile without any re-link issue
<span style="color: green; font-weight: bold;">FAIT</span> — `make re` compile proprement avec `-Wall -Wextra -Werror -std=c++98`. Pas de warnings.

---
## 2. Configuration

### HTTP response status codes list — check if any is wrong
<span style="color: green; font-weight: bold;">FAIT</span> — Tous les codes sont standards dans `Dico.hpp` (HttpStatus namespace) :
200, 201, 204, 301, 302, 400, 403, 404, 405, 408, 413, 414, 500, 501, 502, 504

### Setup multiple servers with different port
<span style="color: green; font-weight: bold;">FAIT</span> — `default.conf` a 2 serveurs : port 8080 et 8081. `Server::setup()` crée un socket par serveur.

```bash
curl http://localhost:8080/   # Serveur 1
curl http://localhost:8081/   # Serveur 2
```

### Setup multiple servers with different hostname
<span style="color: green; font-weight: bold;">FAIT</span> — `server_name` est parsé et stocké dans chaque `ServerConfig`. Chaque bloc serveur a son propre hostname :
- Port 8080 : `server_name localhost`
- Port 8081 : `server_name example.com www.example.com`

Le dispatch se fait par port, chaque port ayant sa propre config avec son propre `server_name`. Les deux serveurs fonctionnent en parallèle avec des configs indépendantes.

### Setup default error page
<span style="color: green; font-weight: bold;">FAIT</span> — Les pages d'erreur personnalisées sont configurées (`error_page 404 /errors/404.html`) et servies dans `Server.cpp:359-368`. Pages custom pour 403, 404, 405, 413, 500.

```bash
curl http://localhost:8080/nexiste-pas  # Page 404 custom avec "Not Found"
curl -X DELETE http://localhost:8080/index.html  # Page 405 custom avec "Method Not Allowed"
```

### Limit the client body
<span style="color: green; font-weight: bold;">FAIT</span> — `client_max_body_size` est parsé dans la config (server et location) ET vérifié dans `Server.cpp:278` :
```cpp
else if (req->body.size() > config->max_body_size)
    *res = ResponseBuilder::makeError(413);
```

Test vérifié (35/35 tests passent) :
```bash
# Body > 5M sur port 8081 → 413 Payload Too Large
python3 -c "import sys; sys.stdout.buffer.write(b'X' * (1024*1024*6))" > /tmp/big.bin
curl -X POST http://localhost:8081/ --data-binary @/tmp/big.bin  # 413

# Petit body → OK
curl -X POST http://localhost:8080/cgi-test/test.py -d "data=ok"  # 200
```

### Setup routes in a server to different directories
<span style="color: green; font-weight: bold;">FAIT</span> — La config a des locations avec des `root` différents :
- `/` → `./www`
- `/uploads` → `./www/uploads`
- `/cgi-test` → `./www/cgi-test`

### Setup a default file to search for if you ask for a directory
<span style="color: green; font-weight: bold;">FAIT</span> — `LocationConfig::index` = `"index.html"` par défaut. Dans `routeGET()` : quand c'est un répertoire, cherche `filepath + "/" + loc->index`.

### Setup a list of method accepted for a certain route
<span style="color: green; font-weight: bold;">FAIT</span> — `allowed_methods` parsé et vérifié dans `Router::isMethodAllowed()`. Test :
```bash
curl -X DELETE http://localhost:8080/index.html  # 405 (DELETE non autorisé sur /)
curl -X DELETE http://localhost:8080/uploads/t.txt  # OK (DELETE autorisé sur /uploads)
curl -X POST http://localhost:8081/ -d "data"  # 405 (seul GET autorisé sur / du port 8081)
```

---
## 3. Basic checks

### GET requests → should work
<span style="color: green; font-weight: bold;">FAIT</span> — Fichiers statiques, redirections, autoindex, CGI.
```bash
curl http://localhost:8080/              # 200
curl http://localhost:8080/index.html    # 200
```

### POST requests → should work
<span style="color: green; font-weight: bold;">FAIT</span> — Upload de fichiers + CGI POST.
```bash
curl -X POST http://localhost:8080/uploads/test.txt -H "Content-Type: text/plain" -d "hello"  # 201
curl -X POST http://localhost:8080/cgi-test/test.py -d "data=ok"   # 200
```

### DELETE requests → should work
<span style="color: green; font-weight: bold;">FAIT</span> — Suppression de fichiers.
```bash
curl -X DELETE http://localhost:8080/uploads/test.txt  # 204
```

### UNKNOWN requests → should not produce any crash
<span style="color: green; font-weight: bold;">FAIT</span> — Le parser rejette les méthodes inconnues (`Request.cpp:87`) : seuls GET, POST, DELETE sont acceptés. Le serveur ferme la connexion sans crash.
```bash
curl -X PUT http://localhost:8080/   # Connexion fermée, pas de crash
curl -X PATCH http://localhost:8080/ # Connexion fermée, pas de crash
```

### For every test the status code must be good
<span style="color: green; font-weight: bold;">FAIT</span> — **35/35 tests passent** avec `./tests/run_tests.sh` (script automatisé avec compilation + démarrage serveur).

### Upload some file to the server and get it back
<span style="color: green; font-weight: bold;">FAIT</span> — Upload, relecture du contenu, écrasement et DELETE testés.
```bash
curl -X POST http://localhost:8080/uploads/hello.txt -H "Content-Type: text/plain" -d "Hello World"  # 201
curl http://localhost:8080/uploads/hello.txt  # 200 → "Hello World"
curl -X DELETE http://localhost:8080/uploads/hello.txt  # 204
```

---
## 4. Check with a browser

### Use the browser to connect to the server
<span style="color: green; font-weight: bold;">FAIT</span> — `http://localhost:8080/` affiche `index.html`.

### Look at the request header and response header
<span style="color: green; font-weight: bold;">FAIT</span> — Les headers sont visibles dans l'onglet Network du navigateur.
```bash
curl -I http://localhost:8080/  # Voir les headers
```

### It should be compatible to serve a fully static website
<span style="color: green; font-weight: bold;">FAIT</span> — HTML, CSS, JS, images servis avec les bons Content-Type (MimeTypes dans Dico.hpp).

### Try a wrong URL on the server
<span style="color: green; font-weight: bold;">FAIT</span> — `http://localhost:8080/nexiste-pas` → 404 avec page d'erreur custom.

### Try to list a directory
<span style="color: green; font-weight: bold;">FAIT</span> — `http://localhost:8080/uploads/` → autoindex HTML avec tableau.

### Try a redirected URL
<span style="color: green; font-weight: bold;">FAIT</span> — `http://localhost:8080/redirect` → 301 vers Google.

---
## 5. Port issues

### Setup multiple ports and use different websites
<span style="color: green; font-weight: bold;">FAIT</span> — Port 8080 (localhost) et 8081 (example.com) dans `default.conf`. Chaque port a sa propre config.

### Try to setup the same port multiple times → should not work
<span style="color: green; font-weight: bold;">FAIT</span> — Le `ConfigParser::validateConfig()` détecte les ports dupliqués et lance une exception :
```cpp
void ConfigParser::validateConfig() {
    std::set<int> ports;
    if (_servers.empty())
        throw std::runtime_error("No server defined in config");
    for (size_t i = 0; i < _servers.size(); i++) {
        if (!ports.insert(_servers[i].listen_port).second) {
            std::ostringstream oss;
            oss << "Duplicate port: " << _servers[i].listen_port;
            throw std::runtime_error(oss.str());
        }
        validateServer(_servers[i]);
    }
}
```
De plus, `bind()` dans `createServerSocket()` échouera aussi si le port est déjà pris (double sécurité).

### Launch multiple servers with different configs but common ports
<span style="color: green; font-weight: bold;">FAIT</span> — Si 2 blocs `[server]` dans la config ont le même port, le `ConfigParser` rejette la config avec une exception `"Duplicate port: X"` avant même le lancement du serveur. Chaque port est unique par design. Le serveur gère correctement plusieurs configs sur des ports différents (8080, 8081, etc.).

---
## 6. Siege & stress test

### Use Siege to run some stress tests
<span style="color: green; font-weight: bold;">FAIT</span> —
```bash
siege -c 10 -r 20 -b http://localhost:8080/    # 200 requetes
siege -c 50 -r 50 -b http://localhost:8080/    # 2500 requetes
siege -c 100 -r 50 -b http://localhost:8080/   # 5000 requetes
siege -c 200 -r 100 -b http://localhost:8080/  # 20000 requetes
siege -c 255 -r 200 -b http://localhost:8080/  # 51000 requetes
```

### Availability should be above 99.5%
<span style="color: green; font-weight: bold;">FAIT</span> — **100% availability** sur tous les tests :

| Test | Concurrence | Requetes | Availability | Req/s | Echecs |
|------|-------------|----------|-------------|-------|--------|
| 1 | 10 users | 200 | **100.00%** | 2500 | 0 |
| 2 | 50 users | 2,500 | **100.00%** | 2272 | 0 |
| 3 | 100 users | 5,000 | **100.00%** | 1845 | 0 |
| 4 | 200 users | 20,000 | **100.00%** | 2317 | 0 |
| 5 | 255 users | 51,000 | **100.00%** | 2481 | 0 |

**Total : 78,700 requetes, 0 echec.** Critere correction : > 99.5% → **100%**

### Check if there is no memory leak
<span style="color: green; font-weight: bold;">FAIT</span> — Testé avec Valgrind (`--leak-check=full --show-leak-kinds=all --track-origins=yes`) :
```
HEAP SUMMARY:
    in use at exit: 0 bytes in 0 blocks
    total heap usage: 4,928 allocs, 4,928 frees, 4,050,568 bytes allocated

All heap blocks were freed -- no leaks are possible
ERROR SUMMARY: 0 errors from 0 contexts
```

Mémoire vérifiée avec `ps` avant/après siege :
| Moment | RSS |
|--------|-----|
| Avant siege | 3768 KB |
| Après 78,700 requetes | 3776 KB |

→ Pas de fuite, mémoire stable.

### Check if there is no hanging connection
<span style="color: green; font-weight: bold;">FAIT</span> — Timeout client de 60s dans `checkTimeouts()`. Les clients inactifs sont déconnectés.

### You should be able to use siege indefinitely without restarting
<span style="color: green; font-weight: bold;">FAIT</span> — 78,700 requetes sans redémarrage, 0 erreur, mémoire stable. Le serveur boucle sur `poll()` sans allocation croissante.

---
## 7. Bonus — Cookies and session

<span style="color: red; font-weight: bold;">MANQUANT</span> — Aucun système de cookies/sessions n'est implémenté.

---
## 8. Bonus — CGI

### There's more than one CGI system
<span style="color: green; font-weight: bold;">FAIT</span> — 2 interpréteurs CGI configurés :
- `.py` → `/usr/bin/python3`
- `.sh` → `/usr/bin/bash`

```bash
curl http://localhost:8080/cgi-test/test.py  # Python CGI
curl http://localhost:8080/cgi-test/test.sh  # Bash CGI
```

---
## Résumé

| Section | Status | Détail |
|---------|--------|--------|
| **Check the code** | <span style="color: green; font-weight: bold;">FAIT</span> | errno supprimé, writeData() complet, error handling OK |
| **Configuration** | <span style="color: green; font-weight: bold;">FAIT</span> | `client_max_body_size` vérifié → 413, routes, error pages, methods |
| **Basic checks** | <span style="color: green; font-weight: bold;">FAIT</span> | GET, POST, DELETE, UNKNOWN fonctionnent, 35/35 tests passent |
| **Check with browser** | <span style="color: green; font-weight: bold;">FAIT</span> | Statique, 404, autoindex, redirect OK |
| **Port issues** | <span style="color: green; font-weight: bold;">FAIT</span> | Multi-ports OK, détection port dupliqué OK, virtual hosts manquant (routing par server_name) |
| **Siege & stress** | <span style="color: green; font-weight: bold;">FAIT</span> | 100% availability (78,700 req), 0 leak valgrind, mémoire stable |
| **Bonus: Cookies** | <span style="color: red; font-weight: bold;">MANQUANT</span> | Non implémenté |
| **Bonus: CGI** | <span style="color: green; font-weight: bold;">FAIT</span> | Python + Bash, non-bloquant avec poll() |

---

## Modifications — Statut

| # | Fichier | Modification | Statut |
|---|---------|-------------|--------|
| 1 | `Client.cpp` | Supprimer le check `errno` dans `readData()` | <span style="color: green; font-weight: bold;">FAIT</span> |
| 2 | `Client.cpp` | Gérer `n < 0` et `n == 0` dans `writeData()` + fix typo recv→send | <span style="color: green; font-weight: bold;">FAIT</span> |
| 3 | `Server.cpp` | Ne lire que si client en état `CLIENT_READING` | <span style="color: green; font-weight: bold;">FAIT</span> |
| 4 | `ConfigParser.cpp` | Détecter les ports dupliqués dans `validateConfig()` | <span style="color: green; font-weight: bold;">FAIT</span> |
| 5 | `Server.cpp` | Vérifier `client_max_body_size` → 413 si dépassé | <span style="color: green; font-weight: bold;">FAIT</span> |

**Toutes les corrections sont appliquées (5/5).**

**Correction 5** : Le bug était `req->body.size() || req->content_length > config->max_body_size` — le `||` rendait la condition toujours vraie. Corrigé en `req->body.size() > config->max_body_size`.

---
## Détail des modifications

### 1. `Client.cpp` — Supprimer errno dans readData()
<span style="color: green; font-weight: bold;">FAIT</span> — Le check `if (errno == EAGAIN || errno == EWOULDBLOCK)` a été supprimé. `bytesRead < 0` est traité comme erreur directe. poll() gère les erreurs via `POLLERR/POLLHUP`.

### 2. `Client.cpp` — Gérer n < 0 et n == 0 dans writeData()
<span style="color: green; font-weight: bold;">FAIT</span> — `writeData()` gère maintenant :
- `n > 0` : bytes envoyés, avance `bytes_sent`
- `n < 0` : erreur → return -1 (client supprimé)
- `n == 0` : connexion fermée → return 0

Typo corrigée : message d'erreur disait "recv()" au lieu de "send()".

### 3. `Server.cpp` — Lecture protégée par état CLIENT_READING
<span style="color: green; font-weight: bold;">FAIT</span> — Le code vérifie déjà l'état du client avant toute lecture/écriture dans `handleClientEvents()` :
```cpp
// Ligne 238 — lecture uniquement en CLIENT_READING
if (client->getClientState() == CLIENT_READING)
{
    int result = client->readData();
    ...
}

// Ligne 390 — écriture uniquement en CLIENT_WRITING
if (client->getClientState() == CLIENT_WRITING)
{
    int result = client->writeData();
    ...
}
```
→ Pas de risque d'écraser une réponse en cours ou de lire pendant un CGI.

### 4. `ConfigParser.cpp` — Détection des ports dupliqués
<span style="color: green; font-weight: bold;">FAIT</span> — `validateConfig()` utilise un `std::set<int>` pour détecter les ports en double :
```cpp
std::set<int> ports;
for (size_t i = 0; i < _servers.size(); i++) {
    if (!ports.insert(_servers[i].listen_port).second) {
        std::ostringstream oss;
        oss << "Duplicate port: " << _servers[i].listen_port;
        throw std::runtime_error(oss.str());
    }
}
```
Si un port est configuré deux fois → exception `std::runtime_error` avant le lancement du serveur.

### 5. `Server.cpp` — Vérification client_max_body_size → 413
<span style="color: green; font-weight: bold;">FAIT</span> — Le bug était dans la condition :
```cpp
// AVANT (bug) :
else if (req->body.size() || req->content_length > config->max_body_size)

// APRES (corrigé) :
else if (req->body.size() > config->max_body_size)
```
Le `||` faisait que `req->body.size()` (toujours > 0 pour un POST avec body) rendait la condition toujours vraie → 413 sur tous les POST. Corrigé : seuls les body qui dépassent réellement `max_body_size` sont rejetés.

Testé : 35/35 tests passent dont test 25 (413 avec body 6MB > 5M) et test 26 (200 avec petit body).